# 06 — Ensemble Aggregation & Robustness Grid

Final notebook. Aggregates all model fits + validation results into headline tables and the full robustness grid per [methodology.md §6-§7](../docs/methodology.md).

**Inputs** (loaded from disk — runs independently of prior notebooks once `02-05` have populated their outputs):
- `data/results/{event}/{window}/{variant}/{model}/fit.pkl` — model fits (from 02)
- `data/validation/*.csv` — validation results (from 03-05)

**Outputs**: headline tables and ensemble plots (in-notebook), saved CSVs in `data/validation/final_*.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import DONOR_POOL_VARIANT
from lib.data import list_fits, load_fit, load_validation_table, save_validation_table
from lib.plotting import plot_ensemble_paths, plot_ensemble_gaps

MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
EVENTS = ['russia', 'hormuz']
WINDOWS = ['preferred', 'extended', 'narrow']
VARIANT = DONOR_POOL_VARIANT

print(f'Loading {len(list_fits())} fits from disk.')

Loading 30 fits from disk.


## Headline table — preferred specification only

Mean post-event gap (%) per model, plus ensemble median + IQR. This is the main result table for the thesis.

In [2]:
headline_rows = []
for event in EVENTS:
    event_gaps = {}
    for model in MODELS:
        fit = load_fit(event, 'preferred', model, variant=VARIANT)
        if fit is None:
            continue
        post = fit['gap'][fit['gap'].index >= fit['t0']]
        if len(post) == 0:
            continue
        mean_gap_pct = float(100 * (np.exp(post.mean()) - 1))
        event_gaps[model] = mean_gap_pct
        headline_rows.append({
            'event': event, 'model': model,
            'mean_post_gap_pct': mean_gap_pct,
            'rmspe_pre_log': fit['rmspe_pre'],
        })
    # Ensemble aggregate
    if event_gaps:
        arr = np.array(list(event_gaps.values()))
        headline_rows.append({
            'event': event, 'model': 'ENSEMBLE_MEDIAN',
            'mean_post_gap_pct': float(np.median(arr)),
            'rmspe_pre_log': np.nan,
            'iqr_lo': float(np.quantile(arr, 0.25)),
            'iqr_hi': float(np.quantile(arr, 0.75)),
            'n_models': len(arr),
        })

headline_df = pd.DataFrame(headline_rows)
save_validation_table(headline_df, 'final_headline')
headline_df.round(3)

,event,model,mean_post_gap_pct,rmspe_pre_log,iqr_lo,iqr_hi,n_models
0,russia,convex_scm,29.597,0.106,NaN,NaN,NaN
1,russia,ascm,13.341,0.106,NaN,NaN,NaN
2,russia,elastic_net,21.694,0.055,NaN,NaN,NaN
3,russia,xgboost,36.807,0.039,NaN,NaN,NaN
4,russia,bayesian_ridge,0.503,0.043,NaN,NaN,NaN
5,russia,ENSEMBLE_MEDIAN,21.694,NaN,13.341,29.597,5.0
6,hormuz,convex_scm,38.554,0.066,NaN,NaN,NaN
7,hormuz,ascm,43.666,0.066,NaN,NaN,NaN
8,hormuz,elastic_net,49.820,0.048,NaN,NaN,NaN
9,hormuz,xgboost,36.829,0.048,NaN,NaN,NaN


## Robustness grid — full appendix table

Every (event, window, model) cell. Shows whether the headline estimate is stable across pre-window choices.

In [3]:
grid_rows = []
for event in EVENTS:
    for window in WINDOWS:
        for model in MODELS:
            fit = load_fit(event, window, model, variant=VARIANT)
            if fit is None:
                continue
            post = fit['gap'][fit['gap'].index >= fit['t0']]
            if len(post) == 0:
                continue
            grid_rows.append({
                'event': event, 'window': window, 'model': model,
                'mean_gap_pct': float(100 * (np.exp(post.mean()) - 1)),
                'rmspe_pre_log': fit['rmspe_pre'],
                'n_post': len(post),
            })

grid_df = pd.DataFrame(grid_rows)
save_validation_table(grid_df, 'final_robustness_grid')
grid_df.round(3)

,event,window,model,mean_gap_pct,rmspe_pre_log,n_post
0,russia,preferred,convex_scm,29.597,0.106,151
1,russia,preferred,ascm,13.341,0.106,151
2,russia,preferred,elastic_net,21.694,0.055,151
3,russia,preferred,xgboost,36.807,0.039,151
4,russia,preferred,bayesian_ridge,0.503,0.043,151
5,russia,extended,convex_scm,35.695,0.216,151
6,russia,extended,ascm,19.671,0.216,151
7,russia,extended,elastic_net,41.131,0.140,151
8,russia,extended,xgboost,38.146,0.051,151
9,russia,extended,bayesian_ridge,-21.647,0.087,151


In [4]:
# Pivot for readability
pivot = grid_df.pivot_table(index=['event', 'model'], columns='window',
                             values='mean_gap_pct')
pivot = pivot.reindex(WINDOWS, axis=1)
pivot.round(2)

window                 preferred  extended  narrow
event  model                                      
hormuz ascm                43.67     38.93   42.77
       bayesian_ridge      43.61     39.56   49.69
       convex_scm          38.55     35.32   43.47
       elastic_net         49.82     39.54   42.12
       xgboost             36.83     37.43   39.15
russia ascm                13.34     19.67   24.99
       bayesian_ridge       0.50    -21.65  -11.34
       convex_scm          29.60     35.70   29.05
       elastic_net         21.69     41.13   34.46
       xgboost             36.81     38.15   35.44

## Post-window sensitivity — delayed-contamination diagnostic

Per [donor_catalog.md §Temporal dimension](../docs/donor_catalog.md) (option 1). The C/M/H audit is *contemporaneous*; some donors marked clean at $T_0$ may acquire a Russia component **over** the post-window (fertilizer cost-push into the soft commodities, discounted-crude re-routing into INR/CNY, terms-of-trade into commodity FX, the inflation→rates path into TLT/HYG). Such **delayed contamination** enters the post-window *projection* — not the pre-window fit — so it pulls the synthetic **up** and biases the gap **toward zero**, with the bias growing as the horizon lengthens.

This re-summarises the **same** preferred-window fits over progressively longer post-windows (1, 2, 3, 6 months, and full). No refitting: weights are learned pre-$T_0$ and unchanged; only the horizon over which the gap is averaged changes. Reading:

- **Monotone decline** with horizon → delayed-contamination signature; the short-horizon estimate is the least-contaminated bound.
- **Flat** → delayed contamination is immaterial; the headline gap is horizon-robust.
- **Rising** → the treatment effect is still building in (the opposite of contamination).

This is the post-window complement to the pre-window robustness grid above, and shares the logic of the OPEC+ truncation ([methodology.md §2](../docs/methodology.md)). Output: `data/validation/final_postwindow_sensitivity.csv`.

In [5]:
# ===== Post-window sensitivity (donor_catalog.md §Temporal dimension, option 1) =====
# Delayed/cumulative donor contamination enters the POST-window projection and biases
# the gap toward zero, worsening with horizon. Re-summarise the SAME pre-window fits
# over progressively longer post-windows -- no refitting (weights are learned pre-T0).
from pandas.tseries.offsets import DateOffset

MONTH_HORIZONS = [1, 2, 3, 6]

sens_rows = []
for event in EVENTS:
    fits = {m: load_fit(event, 'preferred', m, variant=VARIANT) for m in MODELS}
    fits = {m: f for m, f in fits.items() if f is not None}
    if not fits:
        continue
    t0 = next(iter(fits.values()))['t0']
    ref_idx = next(iter(fits.values()))['gap'].index
    full_end = max(f['gap'].index.max() for f in fits.values())

    horizons = [(f'{h}m', t0 + DateOffset(months=h)) for h in MONTH_HORIZONS
                if t0 + DateOffset(months=h) < full_end]
    horizons.append(('full', full_end))

    for label, end in horizons:
        model_gaps = {}
        for m, f in fits.items():
            post = f['gap'][(f['gap'].index >= t0) & (f['gap'].index <= end)]
            if len(post):
                model_gaps[m] = float(100 * (np.exp(post.mean()) - 1))
        arr = np.array(list(model_gaps.values()))
        n_post = int(((ref_idx >= t0) & (ref_idx <= end)).sum())
        row = {'event': event, 'horizon': label, 'horizon_end': str(end.date()),
               'n_post': n_post,
               'ens_median_gap_pct': float(np.median(arr)),
               'iqr_lo': float(np.quantile(arr, 0.25)),
               'iqr_hi': float(np.quantile(arr, 0.75))}
        for m in MODELS:
            row[m] = model_gaps.get(m, np.nan)
        sens_rows.append(row)

sens_df = pd.DataFrame(sens_rows)
save_validation_table(sens_df, 'final_postwindow_sensitivity')

# Diagnosis: compare shortest vs full ensemble median per event.
print('Post-window sensitivity -- ensemble-median gap (%) by horizon:\n')
for event in EVENTS:
    sub = sens_df[sens_df['event'] == event]
    seq = sub['ens_median_gap_pct'].values
    drop = seq[0] - seq[-1]
    if drop > 3:
        verdict = (f'DECLINING by {drop:.1f} pp from {sub.iloc[0]["horizon"]} to full '
                   f'-> delayed-contamination signature (gap attenuates with horizon; '
                   f'short-horizon = least-contaminated bound)')
    elif drop < -3:
        verdict = (f'RISING by {-drop:.1f} pp -> effect still building in '
                   f'(no attenuation; opposite of contamination)')
    else:
        verdict = (f'FLAT (delta={drop:+.1f} pp) -> delayed contamination immaterial; '
                   f'headline gap robust to horizon')
    print(f'{event}: ' + '  '.join(f'{r.horizon}={r.ens_median_gap_pct:.1f}%'
                                   for r in sub.itertuples()))
    print(f'    -> {verdict}\n')

sens_df.round(2)

Post-window sensitivity -- ensemble-median gap (%) by horizon:

russia: 1m=32.1%  2m=23.9%  3m=21.5%  6m=25.0%  full=21.7%
    -> DECLINING by 10.4 pp from 1m to full -> delayed-contamination signature (gap attenuates with horizon; short-horizon = least-contaminated bound)

hormuz: 1m=7.4%  2m=31.6%  full=43.6%
    -> RISING by 36.2 pp -> effect still building in (no attenuation; opposite of contamination)



,event,horizon,horizon_end,n_post,ens_median_gap_pct,iqr_lo,iqr_hi,convex_scm,ascm,elastic_net,xgboost,bayesian_ridge
0,russia,1m,2022-03-24,21,32.08,29.16,33.87,36.32,29.16,33.87,32.08,26.50
1,russia,2m,2022-04-24,40,23.88,18.67,26.13,27.92,18.67,23.88,26.13,14.47
2,russia,3m,2022-05-24,61,21.54,14.70,25.13,25.13,14.70,21.54,28.38,9.76
3,russia,6m,2022-08-24,126,24.98,16.93,32.10,32.10,16.93,24.98,38.79,6.49
4,russia,full,2022-09-30,151,21.69,13.34,29.60,29.60,13.34,21.69,36.81,0.50
5,hormuz,1m,2026-03-01,20,7.36,6.36,10.25,6.36,10.25,13.59,3.57,7.36
6,hormuz,2m,2026-04-01,43,31.60,27.91,33.10,27.91,33.10,39.07,26.40,31.60
7,hormuz,full,2026-04-27,59,43.61,38.55,43.67,38.55,43.67,49.82,36.83,43.61


In [6]:
import plotly.graph_objects as go
from lib.plotting import save_html

for event in EVENTS:
    sub = sens_df[sens_df['event'] == event].reset_index(drop=True)
    x = sub['horizon'].tolist()
    lo, hi = sub['iqr_lo'].tolist(), sub['iqr_hi'].tolist()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x + x[::-1], y=hi + lo[::-1], fill='toself',
                             fillcolor='rgba(31,119,180,0.15)', line=dict(width=0),
                             name='IQR (model spread)', hoverinfo='skip'))
    for m in MODELS:
        if m in sub.columns:
            fig.add_trace(go.Scatter(x=x, y=sub[m], mode='lines', name=m,
                                     line=dict(width=1, dash='dot'), opacity=0.5))
    fig.add_trace(go.Scatter(x=x, y=sub['ens_median_gap_pct'], mode='lines+markers',
                             name='ensemble median', line=dict(color='#1f77b4', width=3)))
    fig.update_layout(title=f'{event.title()} — post-window sensitivity of the gap '
                            f'(declining = delayed-contamination signature)',
                      xaxis_title='post-window horizon', yaxis_title='mean gap (%)',
                      template='plotly_white', height=430,
                      margin=dict(t=70, b=40, l=60, r=40))
    save_html(fig, f'{event}_postwindow_sensitivity', ROOT / 'plots').show()

## Ensemble visualization — preferred window

In [7]:
for event in EVENTS:
    fits = {}
    for model in MODELS:
        f = load_fit(event, 'preferred', model, variant=VARIANT)
        if f is not None:
            fits[model] = f
    if fits:
        plot_ensemble_paths(fits, title=f'{event.title()} — ensemble synthetic paths').show()
        plot_ensemble_gaps(fits, title=f'{event.title()} — ensemble gap (%)').show()

## Validation pass/fail summary (from 03_Validate)

In [8]:
valid_summary = load_validation_table('validation_summary')
if valid_summary is not None:
    print(valid_summary.round(4).to_string())
else:
    print('validation_summary.csv not found — run 03_Validate first.')

    event           model  wf_train_rmse  wf_val_rmse  wf_ratio  pf_mean_pct  pf_slope_yr   pf_r2  drift_contribution_pct
0  russia      convex_scm         0.1094       0.1270    1.1604       0.4285      10.0104  0.2097                  5.8060
1  russia            ascm         0.0431       0.0879    2.0400       0.1140       0.3351  0.0011                  0.1943
2  russia     elastic_net         0.0488       0.0894    1.8311       0.1525       0.9087  0.0061                  0.5270
3  russia         xgboost         0.0391       0.1175    3.0026       0.0768       3.3595  0.1709                  1.9485
4  russia  bayesian_ridge         0.0313       0.0963    3.0731       0.0932       0.1519  0.0003                  0.0881
5  hormuz      convex_scm         0.0593       0.0745    1.2574       0.2611      -7.0425  0.2535                 -1.7606
6  hormuz            ascm         0.0474       0.0639    1.3477       0.1575      -1.9267  0.0290                 -0.4817
7  hormuz     elastic_ne

## Cross-event transfer table (from 05_Cross_Event)

In [9]:
transfer = load_validation_table('cross_event_transfer')
if transfer is not None:
    print(transfer.round(3).to_string())
else:
    print('cross_event_transfer.csv not found — run 05_Cross_Event first.')

            model  russia_rmspe_pre  hormuz_independent_rmspe_pre  hormuz_transferred_rmspe_pre  hormuz_independent_post_gap_pct  hormuz_transferred_post_gap_pct  transferred_minus_independent_pct
0      convex_scm             0.106                         0.066                         0.109                           38.554                           46.960                              8.406
1            ascm             0.106                         0.066                         0.144                           43.666                           27.268                            -16.397
2     elastic_net             0.055                         0.048                         0.185                           49.820                           23.185                            -26.635
3         xgboost             0.039                         0.048                         1.065                           36.829                          -58.171                            -95.000
4  bayesian_rid

## External validation — EIA STEO pre-invasion forecasts (Russia only)

Compare the SCM synthetic counterfactual against the U.S. EIA's last pre-invasion Brent forecasts. Both STEOs were issued *before* the 2022-02-24 invasion, so their forecasts for the post-event window represent EIA's independent counterfactual built on a structural supply/demand model:

- **STEO Jan-22** issued ~2022-01-11 (six weeks pre-invasion)
- **STEO Feb-22** issued ~2022-02-08 (sixteen days pre-invasion, the last STEO before the invasion)

The two methods (SCM cross-asset co-movement vs STEO supply/demand structural model) share no information path. Agreement is evidence that the SCM is not producing a fantasy counterfactual; disagreement is informative about which factor structure each method emphasises.

**Source**: STEO archives at <https://www.eia.gov/outlooks/steo/archives/> (publicly accessible XLSX files). Files **tracked in `data/external/eia/`** — committed verbatim from EIA so provenance is captured even if EIA reorganises their archive. Filenames in the repo match the EIA archive URL filenames (`jan22_base.xlsx`, `feb22_base.xlsx`) for trivial verification. The Brent row is `BREPUUS` (EIA's standard variable code for Brent spot price) in worksheet `2tab`; 2022 monthly values occupy columns 50–61 (the 48-month offset from Jan-2018 baseline).

In [10]:
"""External validation: SCM synthetic vs EIA STEO pre-invasion forecasts (Russia 2022).

Compare the SCM ensemble counterfactual against EIA's two last pre-invasion
forecasts of 2022 Brent. The two methods share no information path (SCM uses
cross-asset co-movement; STEO uses an internal supply/demand model), so
agreement is evidence the SCM is not producing a fantasy counterfactual.

Method: build a daily step-function from each STEO's monthly forecast values
(each daily observation in month M takes the STEO's forecast for M), then
take the mean over the exact SCM treatment day-set (Feb 24 - Sep 30 2022,
151 trading days).

Source files: tracked in data/external/eia/{jan22,feb22}_base.xlsx, copied
verbatim from https://www.eia.gov/outlooks/steo/archives/ (filenames match
the EIA archive URLs so provenance is trivially verifiable). If the tracked
files are missing, the script falls back to downloading from EIA."""
import urllib.request
import openpyxl
import plotly.graph_objects as go
from lib.plotting import save_html

EIA_DIR = ROOT / 'data' / 'external' / 'eia'   # tracked; files committed to repo
EIA_DIR.mkdir(parents=True, exist_ok=True)
STEO_URLS = {
    'jan22': 'https://www.eia.gov/outlooks/steo/archives/jan22_base.xlsx',
    'feb22': 'https://www.eia.gov/outlooks/steo/archives/feb22_base.xlsx',
}

def _load_steo_brent_2022(tag):
    """Return STEO monthly Brent spot forecast for 2022 (12 values, Jan-Dec).

    Filename in repo matches the EIA archive URL filename so anyone can
    verify by downloading the same file directly from EIA.
    """
    p = EIA_DIR / f'{tag}_base.xlsx'
    if not p.exists():
        print(f'  {p.name} not in repo, downloading from {STEO_URLS[tag]}')
        urllib.request.urlretrieve(STEO_URLS[tag], p)
    wb = openpyxl.load_workbook(p, data_only=True)
    for row in wb['2tab'].iter_rows(min_row=1, max_row=40, values_only=True):
        if row[0] == 'BREPUUS':
            # data starts at row[2] = 2018-01; 2022 = +48 months
            return list(row[2 + 48 : 2 + 60])
    raise RuntimeError(f'BREPUUS row not found in {p}')

steo_jan22 = _load_steo_brent_2022('jan22')
steo_feb22 = _load_steo_brent_2022('feb22')

T0 = pd.Timestamp('2022-02-24')
TEND = pd.Timestamp('2022-09-30')

# Use one fit's index for the treatment day-set (actual trading days, holiday-aware)
_ref = load_fit('russia', 'preferred', 'elastic_net', variant=VARIANT)
treatment_days = np.exp(_ref['actual']).loc[T0:TEND].index

def _step_daily(monthly_2022, index):
    s = pd.Series(index=index, dtype=float)
    for m_idx, m_val in enumerate(monthly_2022, start=1):
        mask = (s.index.year == 2022) & (s.index.month == m_idx)
        s.loc[mask] = m_val
    return s

steo_jan22_daily = _step_daily(steo_jan22, treatment_days)
steo_feb22_daily = _step_daily(steo_feb22, treatment_days)

rows = []
for model in MODELS:
    r = load_fit('russia', 'preferred', model, variant=VARIANT)
    if r is None:
        continue
    synth = np.exp(r['synth']).loc[T0:TEND]
    actual = np.exp(r['actual']).loc[T0:TEND]
    rows.append({
        'model': model,
        'n_days': len(treatment_days),
        'actual_mean': actual.mean(),
        'synth_mean': synth.mean(),
        'steo_jan22_mean': steo_jan22_daily.mean(),
        'steo_feb22_mean': steo_feb22_daily.mean(),
    })
steo_compare = pd.DataFrame(rows)
steo_compare['att_synth_pct'] = 100 * (steo_compare['actual_mean'] - steo_compare['synth_mean']) / steo_compare['synth_mean']
steo_compare['att_steo_jan22_pct'] = 100 * (steo_compare['actual_mean'] - steo_compare['steo_jan22_mean']) / steo_compare['steo_jan22_mean']
steo_compare['att_steo_feb22_pct'] = 100 * (steo_compare['actual_mean'] - steo_compare['steo_feb22_mean']) / steo_compare['steo_feb22_mean']
steo_compare['synth_minus_steo_feb22'] = steo_compare['synth_mean'] - steo_compare['steo_feb22_mean']

median_synth = steo_compare['synth_mean'].median()
actual_mean = steo_compare['actual_mean'].iloc[0]
steo_jan22_mean = steo_compare['steo_jan22_mean'].iloc[0]
steo_feb22_mean = steo_compare['steo_feb22_mean'].iloc[0]
steo_compare = pd.concat([steo_compare, pd.DataFrame([{
    'model': 'ENSEMBLE_MEDIAN',
    'n_days': int(steo_compare['n_days'].iloc[0]),
    'actual_mean': actual_mean,
    'synth_mean': median_synth,
    'steo_jan22_mean': steo_jan22_mean,
    'steo_feb22_mean': steo_feb22_mean,
    'att_synth_pct': 100 * (actual_mean - median_synth) / median_synth,
    'att_steo_jan22_pct': 100 * (actual_mean - steo_jan22_mean) / steo_jan22_mean,
    'att_steo_feb22_pct': 100 * (actual_mean - steo_feb22_mean) / steo_feb22_mean,
    'synth_minus_steo_feb22': median_synth - steo_feb22_mean,
}])], ignore_index=True)
save_validation_table(steo_compare, 'external_steo_russia')

print(f'Matched treatment day-set: {T0.date()} - {TEND.date()}  ({len(treatment_days)} trading days)')
print(f'EIA STEO Jan-22 (issued ~2022-01-11): Feb-Sep 2022 step-function mean = ${steo_jan22_mean:.2f}/bbl')
print(f'EIA STEO Feb-22 (issued ~2022-02-08): Feb-Sep 2022 step-function mean = ${steo_feb22_mean:.2f}/bbl')
print()
print(steo_compare.round(2).to_string(index=False))

# Overlay plot: actual + ensemble-median synthetic + STEO step functions
actual_full = np.exp(_ref['actual'])
synth_full = np.exp(_ref['synth'])
fig = go.Figure()
fig.add_trace(go.Scatter(x=actual_full.index, y=actual_full.values, mode='lines',
                         name='Observed Brent', line=dict(color='#111111', width=2)))
fig.add_trace(go.Scatter(x=synth_full.index, y=synth_full.values, mode='lines',
                         name='SCM synthetic (Elastic-net = ensemble median)',
                         line=dict(color='#9467bd', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=steo_feb22_daily.index, y=steo_feb22_daily.values,
                         mode='lines', line_shape='hv',
                         name='EIA STEO Feb-22 (last pre-invasion)',
                         line=dict(color='#2ca02c', width=1.5, dash='dot')))
fig.add_trace(go.Scatter(x=steo_jan22_daily.index, y=steo_jan22_daily.values,
                         mode='lines', line_shape='hv',
                         name='EIA STEO Jan-22',
                         line=dict(color='#ff7f0e', width=1, dash='dot')))
fig.add_vline(x=T0, line_dash='dot', line_color='grey')
fig.add_annotation(x=T0, y=1.02, yref='paper', text='T₀ (invasion)',
                   showarrow=False, font=dict(size=10, color='grey'), xanchor='left')
fig.update_layout(title='Russia 2022 — SCM synthetic vs EIA STEO pre-invasion forecasts',
                  yaxis_title='USD / bbl', xaxis_title='Date',
                  template='plotly_white', hovermode='x unified', height=480,
                  legend=dict(orientation='h', y=-0.18, x=0, xanchor='left', yanchor='top'),
                  margin=dict(t=70, b=80, l=70, r=40))
save_html(fig, 'russia_steo_validation', ROOT / 'plots').show()

Matched treatment day-set: 2022-02-24 - 2022-09-30  (151 trading days)
EIA STEO Jan-22 (issued ~2022-01-11): Feb-Sep 2022 step-function mean = $75.66/bbl
EIA STEO Feb-22 (issued ~2022-02-08): Feb-Sep 2022 step-function mean = $84.94/bbl

          model  n_days  actual_mean  synth_mean  steo_jan22_mean  steo_feb22_mean  att_synth_pct  att_steo_jan22_pct  att_steo_feb22_pct  synth_minus_steo_feb22
     convex_scm     151       108.54       83.55            75.66            84.94          29.90               43.45               27.78                   -1.39
           ascm     151       108.54       95.52            75.66            84.94          13.62               43.45               27.78                   10.58
    elastic_net     151       108.54       88.92            75.66            84.94          22.06               43.45               27.78                    3.98
        xgboost     151       108.54       79.28            75.66            84.94          36.91               43

## Reading the headline result

The headline thesis statement is the **ensemble median for Hormuz** with the IQR as model uncertainty. Russia's ensemble median serves as the **magnitude validation**, anchored two ways:

1. **External**: the SCM ensemble-median counterfactual sits within ~$4/bbl of the EIA STEO Feb-22 step-function forecast over the identical 151-day treatment window (see external-validation section above; ensemble synth $88.92 vs STEO Feb-22 $84.94, Δ +$3.98). The implied ATTs disagree by ~5.7 pp — small enough to read the SCM as internally consistent with an independent structural-model counterfactual issued sixteen days before the invasion.

2. **Internal**: the cross-event weight-transfer table reports whether the Hormuz estimate's confidence should be widened due to regime drift between 2020-22 and 2024-26. Convex SCM and ASCM transfer with plausible pre-RMSPE; the three non-convex models do not — that asymmetry is documented in [validation.md §5f](../docs/validation.md) and inherited by the Hormuz headline.